In [1]:
import os

import nest_asyncio
from rich.progress import track

from sdg.configs.generators import QuestionVerbatimAnswerGeneratorConfig

nest_asyncio.apply()

from rich.console import Console

console = Console()

In [ ]:
to_process_en = {
    "drama": [
        # "drie015",
        "lois006",
        "peen001"
    ],
    "jeugdliteratuur": [
        "sche039"
    ],
    "poezie": [
        "lenn006"
    ],
    "proza": [
        "_kle007",
        "_kon002",
        "pier003"
    ]
}

to_process_nl = {
    "jeugdliteratuur": [
        "goej001",
        "hoff049"
    ],
    "poezie": [
        "beer008",
        "dool003"
    ],
}

to_process = to_process_en

original_column_information = {
    "title_column": "titel",
    "first_name_column": "voornaam",
    "last_name_column": "achternaam",
    "prefix_name_column": "voorvoegsel",
    "text_column": "text"
}

language_column_information = {
    # "Nederlands": {
    #     "title_column": "titel",
    #     "first_name_column": "voornaam",
    #     "last_name_column": "achternaam",
    #     "prefix_name_column": "voorvoegsel",
    #     "text_column": "modern_nederlands_translation"
    # },
    "english": {
        "title_column": "titel",
        "first_name_column": "voornaam",
        "last_name_column": "achternaam",
        "prefix_name_column": "voorvoegsel",
        "text_column": "english_translation"
    }
}

DATASET_PATH = f"data/translated/splitted_1850_text_length_max_70k_top_5_most_published"
SAVE_PATH = f"data/qa/selected_books"

os.makedirs(SAVE_PATH, exist_ok=True)

In [ ]:
from psalm.core.models import Book
from typing import Optional
from sdg.validations import VerbatimQAList


def convert_to_book(row: dict[str, str], column_information: dict[str, str], language: str) -> Book:
    title = row[column_information["title_column"]]

    if "author_column" in column_information:
        author = row[column_information["author_column"]]
    elif row[column_information["last_name_column"]] is None or row[column_information["first_name_column"]] == row[column_information["last_name_column"]]:
        author = row[column_information["first_name_column"]]
    else:
        first_name: str = row[column_information["first_name_column"]]
        last_name: str = row[column_information["last_name_column"]]
        prefix: Optional[str] = row[column_information["prefix_name_column"]]

        author = f"{first_name}{' ' + prefix if prefix is not None and prefix.strip() else ''} {last_name}"

    text = row[column_information["text_column"]]

    return Book(title=title, author=author, text=text, language=language)

In [4]:
from sdg.validations import BookTranslations


def convert_to_book_translations(row: dict[str, str], original_column_information: dict[str, str], translation_columns_information: dict[str, dict[str, str]]) -> BookTranslations:
    original_book = convert_to_book(row, original_column_information, language="Oud Nederlands (1850-1900)")
    translations = []

    for language, column_info in translation_columns_information.items():
        translations.append(convert_to_book(row, column_info, language=language))

    return BookTranslations(
        original=original_book,
        translations=translations
    )

In [5]:
from sdg.generator import QuestionVerbatimAnswerGenerator
import polars as pl

def process(author, genre):
    import os

    dataset_genre_path = os.path.join(DATASET_PATH, genre)
    dataset_author_path = os.path.join(dataset_genre_path, f"{author}.parquet")
    df = pl.read_parquet(dataset_author_path)

    print(f"Length: {len(df)}")

    from sdg.configs import ModelConfig

    generator_config = QuestionVerbatimAnswerGeneratorConfig(
        min_amount_of_questions_per_book=1,
        max_amount_of_questions_per_book=9
    )

    model_config = ModelConfig(
        model_name="gpt-4o-mini",
        temperature=0.1,
        timeout=300,
        max_retries=3
    )

    generator = QuestionVerbatimAnswerGenerator(generator_config, model_config)

    final_generated_qas = {}
    print(df["ti_id"].to_list())
    
    for row in track(df.iter_rows(named=True), total=len(df), description=f"{author}...", transient=True):
        ti_id = row["ti_id"]
        book_translations = convert_to_book_translations(row, original_column_information, language_column_information)

        print(f"Processing: {ti_id}")

        for translation in track(book_translations.translations, total=len(book_translations.translations), description=f"{ti_id}...", transient=True):
            generated_qas = generator.generate_with_splitting(translation, 10000)

            if translation.language not in final_generated_qas:
                final_generated_qas[translation.language] = {}

            if ti_id not in final_generated_qas[translation.language]:
                final_generated_qas[translation.language][ti_id] = {}

            final_generated_qas[translation.language][ti_id] = generated_qas
            print(f"{ti_id} processed")

    for language, ti_ids in track(final_generated_qas.items(), total=len(final_generated_qas.items()), description=f"Saving QA dataset for {genre}...", transient=True):
        ids = []
        questions = []
        answers = []
        genre_list = []
        author_list = []

        print("Saving:")
        print(ti_ids.keys())

        for ti_id, qas in ti_ids.items():
            qas: VerbatimQAList = qas

            print(f"saving {ti_id}")
            
            for qa in qas.items:
                ids.append(ti_id)
                questions.append(qa.question)
                answers.append(qa.answer)
                genre_list.append(genre)
                author_list.append(author)

        new_df = pl.DataFrame({
            "ti_id": ids,
            "author": author_list,
            "genre": genre_list,
            "question": questions,
            "answer": answers
        })

        save_genre_path = os.path.join(SAVE_PATH, genre)
        save_language_path = os.path.join(save_genre_path, language)
        save_author_path = os.path.join(save_language_path, f"qa_{author}.parquet")

        os.makedirs(save_language_path, exist_ok=True)
        new_df.write_parquet(save_author_path)

In [6]:
failed_count = 0
failed_info = {}

for genre, authors in track(to_process.items(), total=len(to_process.items()), description=f"Generating QAs for Corpus...", transient=True):
    for author in track(authors, total=len(authors), description=f"{genre}...", transient=True):
        try:
            process(author, genre)
        except Exception as e:
            console.print_exception()
            failed_count += 1
            if genre not in failed_info:
                failed_info[genre] = {}
            if author not in failed_info[genre]:
                failed_info[genre][author] = {"count": 0, "errors": []}
            failed_info[genre][author]["count"] += 1
            failed_info[genre][author]["errors"].append(str(e))

Output()

Length: 2

['drie015geen01', 'drie015oude01']

Processing: drie015geen01

Chunks... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:02:13

...

drie015oude01 processed

Saving:

dict_keys(['drie015geen01', 'drie015oude01'])

saving drie015geen01

saving drie015oude01

In [7]:
failed_info

{}